# Week 3 follow-up #2: full dataset + discriminative LR + per-epoch checkpoints (Kaggle GPU)

**Why this run exists:** configs D/E (unfreeze last 2 vision layers, 3 epochs, same 1500-image
subset, same LR for vision and decoder) overfit -- train loss dropped 1.99 -> 0.85 but BLEU
dropped 27.2 -> 23.4 on held-out images. This run fixes the training recipe:

1. **Nearly all the data**: ~7,791 training images (vs. 1,500).
2. **Discriminative learning rate**: vision layers at 5e-6, decoder at 5e-5 (10x lower for the
   pretrained backbone).
3. **Checkpoint + evaluate after every epoch** (3 epochs, greedy and beam each time) against a
   **validation** set -- gives an actual train-loss-vs-eval-BLEU curve instead of guessing an
   epoch count.

**Also fixes a methodology gap**: every config so far (zero-shot, A-E) has been compared against
the *same* 100 held-out images. Comparing many configs against one fixed set and picking the best
is a form of validation-set leakage -- the reported score for whichever config "wins" gets
optimistic. So this run adds a proper 3-way split:
- **Train** (7,791 images) -- updates the weights
- **Validation** (the same 100 images every prior config used) -- used to pick the best epoch,
  exactly as before
- **Test** (200 *new* images, never touched during training or epoch selection) -- evaluated
  exactly once, only for the final chosen epoch, as an honest, non-cherry-picked number

## Setup
1. New Kaggle Notebook, paste this file in.
2. **Settings -> Accelerator -> GPU** (T4 x2 or P100).
3. **Add Data** -> `adityajn105/flickr8k`.
4. Run all cells. Estimated ~50-65 minutes (3 epochs x ~15-20 min + 6 validation eval passes +
   2 final test eval passes).
5. Download `week3_finetune_fulldata_results.json`/`.csv` (has both validation-per-epoch AND the
   final test-set numbers) from the Output tab, plus the best epoch's checkpoint
   (`config_F_epoch{N}_checkpoint/`), into this repo's `results/` folder.

### 1. Setup

In [ ]:
import os
import json
import time
import random
import pandas as pd
import torch
import evaluate
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import BlipProcessor, BlipForConditionalGeneration

KAGGLE_INPUT_DIR = "/kaggle/input/flickr8k"
OUTPUT_DIR = "/kaggle/working"

CAPTIONS_PATH = os.path.join(KAGGLE_INPUT_DIR, "captions.txt")
IMG_DIR = os.path.join(KAGGLE_INPUT_DIR, "Images")

device = "cuda" if torch.cuda.is_available() else "cpu"
assert device == "cuda", "No GPU detected -- check Settings > Accelerator on Kaggle."
print(f"Using device: {device}")

MODEL_NAME = "Salesforce/blip-image-captioning-base"

df = pd.read_csv(CAPTIONS_PATH)
df.columns = ["image", "caption"]
print(f"Total captions: {len(df)}, unique images: {df['image'].nunique()}")

### 2. Train / validation / test split

Validation = the same first-100 images every prior config (A-E) used, for continuity. Test = a
*new* 200-image slice, held out from everything until the very end. Train = everything else.

In [ ]:
random.seed(42)
all_images = df["image"].drop_duplicates().tolist()
random.shuffle(all_images)

val_images = all_images[:100]           # same as every prior config's "eval" set
test_images = all_images[100:300]       # NEW: held out, touched exactly once, at the end
train_images = all_images[300:]         # everything else -- ~7791 images

train_df = df[df["image"].isin(train_images)].groupby("image").head(2).reset_index(drop=True)
val_refs = [df[df["image"] == img]["caption"].tolist() for img in val_images]
test_refs = [df[df["image"] == img]["caption"].tolist() for img in test_images]

print(f"Training pairs: {len(train_df)} (from {len(train_images)} images)")
print(f"Validation images: {len(val_images)} (used to pick the best epoch, matches configs A-E)")
print(f"Test images: {len(test_images)} (untouched until the final cell)")

### 3. Dataset / collator (batch size raised to 16 for throughput on the larger dataset)

In [ ]:
processor = BlipProcessor.from_pretrained(MODEL_NAME)

class FlickrFineTuneDataset(Dataset):
    def __init__(self, dataframe, img_dir):
        self.df = dataframe.reset_index(drop=True)
        self.img_dir = img_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(os.path.join(self.img_dir, row["image"])).convert("RGB")
        return image, row["caption"]

def collate_fn(batch):
    images, captions = zip(*batch)
    inputs = processor(
        images=list(images), text=list(captions),
        padding="max_length", truncation=True, max_length=32, return_tensors="pt",
    )
    labels = inputs["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    inputs["labels"] = labels
    return inputs

train_dataset = FlickrFineTuneDataset(train_df, IMG_DIR)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
print(f"{len(train_loader)} batches/epoch")

### 4. Eval function (takes an explicit image/ref set -- reused for both validation and the final test pass) + discriminative-LR training with per-epoch checkpointing

In [ ]:
@torch.no_grad()
def evaluate_model(model, images, refs, decoding="greedy"):
    model.eval()
    gen_kwargs = {"max_new_tokens": 30}
    gen_kwargs["num_beams"] = 4 if decoding == "beam" else 1

    predictions = []
    for img_name in tqdm(images, desc=f"eval decoding={decoding}"):
        image = Image.open(os.path.join(IMG_DIR, img_name)).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)
        out = model.generate(**inputs, **gen_kwargs)
        predictions.append(processor.decode(out[0], skip_special_tokens=True))

    bleu = evaluate.load("sacrebleu").compute(predictions=predictions, references=refs)
    rouge = evaluate.load("rouge").compute(predictions=predictions, references=refs)
    return {
        "bleu": bleu["score"],
        "rouge1": rouge["rouge1"] * 100,
        "rouge2": rouge["rouge2"] * 100,
        "rougeL": rouge["rougeL"] * 100,
    }, predictions


def train_with_checkpoints(lr_vision, lr_decoder, num_epochs, unfreeze_last_n_vision_layers):
    model = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)

    num_vision_layers = len(model.vision_model.encoder.layers)
    freeze_up_to = num_vision_layers - unfreeze_last_n_vision_layers
    for i, layer in enumerate(model.vision_model.encoder.layers):
        for p in layer.parameters():
            p.requires_grad = i >= freeze_up_to
    for p in model.vision_model.embeddings.parameters():
        p.requires_grad = False

    vision_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("vision_model")]
    other_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("vision_model")]
    print(f"Vision params trainable: {sum(p.numel() for p in vision_params):,} @ lr={lr_vision}")
    print(f"Other (decoder) params trainable: {sum(p.numel() for p in other_params):,} @ lr={lr_decoder}")

    optimizer = torch.optim.AdamW([
        {"params": vision_params, "lr": lr_vision},
        {"params": other_params, "lr": lr_decoder},
    ])

    epoch_results = {}
    for epoch in range(num_epochs):
        model.train()
        losses = []
        for batch in tqdm(train_loader, desc=f"train epoch {epoch + 1}/{num_epochs}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            losses.append(loss.item())

        # VALIDATION only -- test set stays untouched here
        metrics_greedy, _ = evaluate_model(model, val_images, val_refs, decoding="greedy")
        metrics_beam, _ = evaluate_model(model, val_images, val_refs, decoding="beam")

        avg_loss = sum(losses) / len(losses)
        print(f"Epoch {epoch + 1}: avg_train_loss={avg_loss:.3f}, final_train_loss={losses[-1]:.3f}, "
              f"val_greedy_bleu={metrics_greedy['bleu']:.2f}, val_beam_bleu={metrics_beam['bleu']:.2f}")

        epoch_results[f"epoch{epoch + 1}"] = {
            "avg_train_loss": avg_loss,
            "final_train_loss": losses[-1],
            "val_greedy": metrics_greedy,
            "val_beam": metrics_beam,
        }
        model.save_pretrained(os.path.join(OUTPUT_DIR, f"config_F_epoch{epoch + 1}_checkpoint"))

    return epoch_results

### 5. Run it: 3 epochs, full data, discriminative LR

In [ ]:
t0 = time.time()
epoch_results = train_with_checkpoints(
    lr_vision=5e-6, lr_decoder=5e-5, num_epochs=3, unfreeze_last_n_vision_layers=2,
)
print(f"Total training time: {time.time() - t0:.1f}s")

val_rows = []
for epoch_name, r in epoch_results.items():
    for decoding in ["greedy", "beam"]:
        val_rows.append({
            "config": f"F_{epoch_name}_{decoding}", "epoch": epoch_name, "decoding": decoding,
            "avg_train_loss": r["avg_train_loss"], "final_train_loss": r["final_train_loss"],
            **r[f"val_{decoding}"],
        })
val_results_df = pd.DataFrame(val_rows).set_index("config")
val_results_df

### 6. Pick the best epoch by VALIDATION BLEU, then evaluate it ONCE on the held-out TEST set

This is the only place the test set gets touched -- one model, one evaluation, no cherry-picking.

In [ ]:
best_config = val_results_df["bleu"].astype(float).idxmax()
best_epoch = val_results_df.loc[best_config, "epoch"]
print(f"Best config on VALIDATION: {best_config} (epoch={best_epoch})")

best_model = BlipForConditionalGeneration.from_pretrained(
    os.path.join(OUTPUT_DIR, f"config_F_{best_epoch}_checkpoint")
).to(device)

test_metrics_greedy, _ = evaluate_model(best_model, test_images, test_refs, decoding="greedy")
test_metrics_beam, _ = evaluate_model(best_model, test_images, test_refs, decoding="beam")

print(f"\nFinal TEST set results for {best_epoch} (n={len(test_images)}, never used for model selection):")
print(f"  greedy: {test_metrics_greedy}")
print(f"  beam:   {test_metrics_beam}")

test_results = {
    "best_epoch": best_epoch,
    "best_config_by_validation": best_config,
    "test_set_size": len(test_images),
    "test_greedy": test_metrics_greedy,
    "test_beam": test_metrics_beam,
}

### 7. Save results for download

In [ ]:
val_results_df.to_csv(os.path.join(OUTPUT_DIR, "week3_finetune_fulldata_validation_results.csv"))

with open(os.path.join(OUTPUT_DIR, "week3_finetune_fulldata_results.json"), "w") as f:
    json.dump({
        "validation_per_epoch": json.loads(val_results_df.reset_index().to_json(orient="records")),
        "final_test_results": test_results,
    }, f, indent=2)

print(f"Best epoch by validation: {best_epoch} ({best_config})")
print(f"Compare BLEU -- config A=27.2, C=28.9, D=23.4, E=24.6 (all on the same validation set)")
print(f"This run's validation BLEU: {val_results_df.loc[best_config, 'bleu']:.2f}")
print(f"This run's HONEST test-set BLEU: greedy={test_metrics_greedy['bleu']:.2f}, beam={test_metrics_beam['bleu']:.2f}")
print()
print("Download week3_finetune_fulldata_results.json/csv from the Output tab, plus ONLY the")
print(f"checkpoint folder for the best epoch (config_F_{best_epoch}_checkpoint/) into this repo's results/ folder.")